---
numbering: false
---

# 5.3. Singular Value Decomposition

:::{danger} This note is under construction, and is not yet complete!
:::

Eigenvalues and eigenvectors are powerful concepts, but unfortunately, they only apply to square matrices. It would be nice if we could extend some of their utility to non-square matrices, like matrices containing real data, which typically have many more rows than columns.

This is precisely where the **singular value decomposition** (SVD) comes in. You should think of it as a generalization of the eigenvalue decomposition, $A = V \Lambda V^{-1}$, to non-square matrices.

<!-- The SVD has several applications, but perhaps the most interesting application is to **dimensionality reduction**. -->

In [189]:
# import seaborn as sns
# import numpy as np
# import matplotlib.pyplot as plt
# from sklearn.linear_model import LinearRegression
# from sklearn.decomposition import PCA

# # Load and prepare the data
# penguins = sns.load_dataset("penguins").dropna()

# # Set random seed for reproducibility
# np.random.seed(42)

# # Create a new feature as bill_length_mm * 2 + random gaussian noise
# penguins["noisy_length"] = (
#     2.0 * penguins["bill_length_mm"] +
#     np.random.normal(0, 5, size=len(penguins))
# )

# # Prepare X and y for linear regression
# X = penguins["bill_length_mm"].values.reshape(-1, 1)
# y = penguins["noisy_length"].values

# # Fit linear regression
# reg = LinearRegression()
# reg.fit(X, y)
# slope = reg.coef_[0]
# intercept = reg.intercept_

# # Stack data for PCA
# data_for_pca = np.column_stack([X.flatten(), y])
# pca = PCA(n_components=1)
# center = data_for_pca.mean(axis=0)
# pca.fit(data_for_pca - center)

# # Calculate PCA line direction
# direction = pca.components_[0]
# # Make sure the direction "points right" roughly for nice plotting
# if direction[0] < 0:
#     direction *= -1

# # Create line for plotting
# x_vals = np.linspace(X.min(), X.max(), 200)
# # Linear regression line
# y_reg = slope * x_vals + intercept

# # PCA rank-1 approximation line: passes through mean, along principal direction
# t = x_vals - center[0]
# y_pca = center[1] + direction[1]/direction[0] * t

# # Plotting
# plt.figure(figsize=(9,7))
# plt.scatter(penguins["bill_length_mm"], penguins["noisy_length"],
#             alpha=0.7, c="grey", edgecolor="k", label="Data")

# plt.plot(x_vals, y_reg, color="orange", lw=2.5, label="Linear Regression line")
# plt.plot(x_vals, y_pca, color="gold", lw=2.5, label="PCA Rank 1 Approximation")

# plt.xlabel("Bill Length (mm)", fontsize=13)
# plt.ylabel("Noisy Length = Bill Length × 2 + noise", fontsize=13)
# plt.title("Bill Length vs. Noisy Length with Linear Regression (orange) and PCA (gold)", fontsize=14, pad=12)
# plt.legend()
# plt.grid(True, which='both', color='#f0f0f0')
# plt.tight_layout()
# plt.show()


---

## The Decomposition

It will take a while to develop the SVD, so let me start by telling you the main result.

:::{note} Definition: Singular Value Decomposition
Suppose $X$ is any $n \times d$ matrix (that is, **not necessarily square**). Then there exists a factorization of the form

$$X = U \Sigma V^T$$

where:
- $U$ is an $n \times n$ orthogonal matrix
- $\mathcal{\Sigma}$ is an $n \times d$ diagonal matrix with non-negative real numbers on the diagonal and zeros elsewhere
- $V$ is a $d \times d$ orthogonal matrix

The diagonal entries of $\mathcal{\Sigma}$ are called the **singular values** of $X$. Typically, the singular values are sorted in decreasing order, i.e. $\sigma_1 \geq \sigma_2 \geq \cdots \geq \sigma_r > 0$, where $r = \text{rank}(X)$.
:::

Firstly, note that the SVD has no restrictions on what $X$ is: it can be non-square, and it doesn't even need to be full rank. But note that unlike in the eigenvalue decomposition, where we decomposed $A$ using just one eigenvector matrix $V$ and a diagonal matrix $\mathcal{\Lambda}$, here we need to use two **singular vector** matrices $U$ and $V$ and a diagonal matrix $\mathcal{\Sigma}$.

There's a lot of notation and new concepts above. Let's start with an example, then try and understand where each piece in $X = U \Sigma V^T$ comes from and means. Suppose

$$
X = \begin{bmatrix}
3 & 2 & 5 \\
2 & 3 & 5 \\
2 & -2 & 0 \\
5 & 5 & 10
\end{bmatrix}
$$

$X$ is a $4 \times 3$ matrix with $\text{rank}(X) = 2$, since its third column is the sum of the first two. Its singular value decomposition is given by $X = U \Sigma V^T$:

$$\underbrace{\begin{bmatrix}
\frac{1}{\sqrt{6}} & \frac{1}{3\sqrt{2}} & -1/\sqrt{3} & -2/3 \\
\frac{1}{\sqrt{6}} & -\frac{1}{3\sqrt{2}} & -1/\sqrt{3} & 2/3 \\
0 & \frac{2\sqrt{2}}{3} & 0 & 1/3 \\
\frac{2}{\sqrt{6}} & 0 & 1/\sqrt{3} & 0
\end{bmatrix}}_{U} \underbrace{\begin{bmatrix} 15 & 0 & 0 \\ 0 & 3 & 0 \\ 0 & 0 & 0 \\ 0 & 0 & 0 \end{bmatrix}}_{\Sigma} \underbrace{\begin{bmatrix} \frac{1}{\sqrt{6}} & \frac{1}{\sqrt{6}} & \frac{2}{\sqrt{6}} \\ \frac{1}{\sqrt{2}} & -\frac{1}{\sqrt{2}} & 0 \\ \frac{1}{\sqrt{3}} & \frac{1}{\sqrt{3}} & -\frac{1}{\sqrt{3}} \end{bmatrix}}_{V^T}$$

Two important observations:

1. $U$ and $V$ are both orthogonal matrices, meaning $U^TU = UU^T = I_{4 \times 4}$ and $V^TV = VV^T = I_{3 \times 3}$.
1. $\mathcal{\Sigma}$ contains the singular values of $X$ on the diagonal, arranged in decreasing order. We have that $\sigma_1 = 15$, $\sigma_2 = 3$, and $\sigma_3 = 0$. $X$ has three singular values, but only two are non-zero. **In general, the number of non-zero singular values is equal to the rank of $X$**.

Where did all of these numbers come from?

---

## The Role of $X^TX$ and $XX^T$

The SVD of $X$ depends heavily on the matrices $X^TX$ and $XX^T$. While $X$ itself is $4 \times 3$, 
- $X^TX$ is a **symmetric $3 \times 3$ matrix**, containing the dot products of $X$'s columns
- $XX^T$ is a **symmetric $4 \times 4$ matrix**, containing the dot products of $X$'s rows

Since $X^TX$ and $XX^T$ are both square matrices, they have eigenvalues and eigenvectors. And since they're both symmetric, their eigenvectors for different eigenvalues are orthogonal to each other, as the spectral theorem $A = Q \Lambda Q^T$ guarantees for any symmetric matrix $A$. 

### Singular Values and Singular Vectors

The singular value decomposition involves creatively using the eigenvalues and eigenvectors of $X^TX$ and $XX^T$. Suppose $X = U \Sigma V^T$ is the SVD of $X$. Then, using the facts that $U^TU = I$ and $V^TV = I$, we have:

$$X^TX = (U \Sigma V^T)^T (U \Sigma V^T) = V \Sigma^T \underbrace{U^T U}_{I} \Sigma V^T = \underbrace{V \Sigma^T \Sigma V^T}_{\text{looks like } Q \Lambda Q^T}$$

$$XX^T = (U \Sigma V^T) (U \Sigma V^T)^T = U \Sigma \underbrace{V^T V}_{I} \Sigma^T U^T = \underbrace{U \Sigma \Sigma^T U^T}_{\text{looks like } P \Lambda P^T}$$

**This just looks like we diagonalized $X^TX$ and $XX^T$!** This is saying that:
- $V$ contains the eigenvectors of $X^TX$; $V$'s columns are called the **right singular vectors** of $X$
- $U$ contains the eigenvectors of $XX^T$; $U$'s columns are called the **left singular vectors** of $X$

$X^TX$ and $XX^T$ usually have different sets of eigenvectors, which is why $U$ and $V$ are generally not the same matrix (they don't even have the same shape).

The eigen**values** of $X^TX$ and $XX^T$ are the same, though: those are the non-zero entries of $\Sigma^T \Sigma$ and $\Sigma \Sigma^T$. Since $\mathcal{\Sigma}$ is an $n \times d$ matrix, $\Sigma^T \Sigma$ is a $d \times d$ matrix and $\Sigma \Sigma^T$ is an $n \times n$ matrix. But, when you work out both products, you'll notice that their non-zero values are the same.

Suppose for example that $\Sigma = \begin{bmatrix} \sigma_1 & 0 & 0 \\ 0 & \sigma_2 & 0 \\ 0 & 0 & \sigma_3 \\ 0 & 0 & 0 \end{bmatrix}$. Then,

$$\Sigma^T \Sigma = \begin{bmatrix} \sigma_1 & 0 & 0 & 0 \\ 0 & \sigma_2 & 0 & 0 \\ 0 & 0 & \sigma_3 & 0 \end{bmatrix} \begin{bmatrix} \sigma_1 & 0 & 0 \\ 0 & \sigma_2 & 0 \\ 0 & 0 & \sigma_3 \\ 0 & 0 & 0 \end{bmatrix} = \begin{bmatrix} \sigma_1^2 & 0 & 0 \\ 0 & \sigma_2^2 & 0 \\ 0 & 0 & \sigma_3^2\end{bmatrix}$$

$$\Sigma \Sigma^T = \begin{bmatrix} \sigma_1 & 0 & 0 \\ 0 & \sigma_2 & 0 \\ 0 & 0 & \sigma_3 \\ 0 & 0 & 0 \end{bmatrix} \begin{bmatrix} \sigma_1 & 0 & 0 & 0 \\ 0 & \sigma_2 & 0 & 0 \\ 0 & 0 & \sigma_3 & 0 \end{bmatrix} = \begin{bmatrix} \sigma_1^2 & 0 & 0 & 0 \\ 0 & \sigma_2^2 & 0 & 0 \\ 0 & 0 & \sigma_3^2 & 0 \\ 0 & 0 & 0 & 0 \end{bmatrix}$$

But, these matrices with the squared terms are precisely the $\mathcal{\Lambda}$'s in the spectral decompositions of $X^TX = Q \Lambda Q^T$ and $XX^T = P \Lambda P^T$. This means that $$\sigma_i^2 = \lambda_i \implies \sigma_i = \sqrt{\lambda_i}$$

where $\sigma_i$ is a **singular value** of $X$ and $\lambda_i$ is an eigenvalue of $X^TX$ or $XX^T$.

The above derivation is enough to justify that the eigenvalues of $X^TX$ and $XX^T$ are never negative, but for another perspective, note that both $X^TX$ and $XX^T$ are **positive semidefinite**, meaning their eigenvalues are non-negative. (You're wrestling with this fact in Lab 11 and Homework 10.)

:::{attention} The relationship between singular values/vectors and eigenvalues/vectors

To summarize: in $X = U \Sigma V^T$,
- The columns of $U$ – called the **left singular vectors** of $X$ – are the eigenvectors of $XX^T$
- The columns of $V$ – called the **right singular vectors** of $X$ – are the eigenvectors of $X^TX$
- The **singular values** of $X$, which live on the diagonal of $\mathcal{\Sigma}$, are the **square roots of the eigenvalues** of $X^TX$ and $XX^T$: $$\sigma_i = \sqrt{\lambda_i}$$
:::


:::{tip} Another proof that $X^TX$ and $XX^T$ have the same non-zero eigenvalues
:class: dropdown

Above, we implicitly used the fact that $X^TX$ and $XX^T$ have the same non-zero eigenvalues. Here's a proof of this fact that has nothing to do with the SVD.

Suppose $\vec v_i$ is an eigenvector of $X^TX$ with eigenvalue $\lambda_i$.

$$X^TX \vec v_i = \lambda_i \vec v_i$$

What happens if we multiply both sides on the left by $X$?

$$XX^TX \vec v_i = X \lambda_i \vec v_i$$

Creatively adding parentheses gives us

$$XX^T (X \vec v_i) = \lambda_i (X \vec v_i)$$

This shows that $X \vec v_i$ is an eigenvector of $XX^T$ with eigenvalue $\lambda_i$. The important thing is that the eigenvalue is shared. This logic can be reversed too, to show that if $\vec u_i$ is an eigenvector of $XX^T$ with eigenvalue $\lambda_i$, then $\lambda_i$ is also an eigenvalue of $X^TX$.
:::

### Computing the SVD

To find $U$, $\mathcal{\Sigma}$, and $V^T$, we don't actually need to compute both $X^TX$ and $XX^T$: all of these quantities can be uncovered just with one of them.

Let's return to our example, $X = \begin{bmatrix} 3 & 2 & 5 \\ 2 & 3 & 5 \\ 2 & -2 & 0 \\ 5 & 5 & 10 \end{bmatrix}$. Using what we've just learned, let's find $U$, $\mathcal{\Sigma}$, and $V^T$ ourselves. $X^TX$ has fewer entries than $XX^T$, so let's start with it. I will delegate _some_ number crunching to `numpy`.

In [156]:
X = np.array([[3, 2, 5],
              [2, 3, 5],
              [2, -2, 0],
              [5, 5, 10]])
X.T @ X

array([[ 42,  33,  75],
       [ 33,  42,  75],
       [ 75,  75, 150]])

$X^TX$ is a $3 \times 3$ matrix, but its rank is 2, meaning it will have an eigenvalue of $0$. What are its other eigenvalues?

In [158]:
np.set_printoptions(precision=0, suppress=True)

In [162]:
eigvals, eigvecs = np.linalg.eig(X.T @ X)
eigvals

array([225.,   9.,   0.])

The eigenvalues of $X^TX$ are $225$, $9$, and $0$. This tells us that the singular values of $X$ are $\sqrt{225} = 15$, $\sqrt{9} = 3$, and $0$. So far, we've discovered that $\mathcal{\Sigma} = \begin{bmatrix} 15 & 0 & 0 \\ 0 & 3 & 0 \\ 0 & 0 & 0 \\ 0 & 0 & 0 \end{bmatrix}$. Remember that $\mathcal{\Sigma}$ always has the same shape as $X$ (both $n \times d$), and all of its entries are 0 except for the singular values, which are arranged in decreasing order on the diagonal.

Let's now find the eigenvectors of $X^TX$ – that is, the right singular vectors of $X$ – which we should store in $V$. **We expect the eigenvectors $\vec v_i$ of $X^TX$to be orthogonal**, since $X^TX$ is symmetric.

In [178]:
X.T @ X

array([[ 42,  33,  75],
       [ 33,  42,  75],
       [ 75,  75, 150]])

- For $\lambda_1 = 225$, one eigenvector is $\vec v_1 = \begin{bmatrix} 1 \\ 1 \\ 2 \end{bmatrix}$, since

$$\underbrace{\begin{bmatrix} 42 & 33 & 75 \\ 33 & 42 & 75 \\ 75 & 75 & 150 \end{bmatrix}}_{X^TX} \begin{bmatrix} 1 \\ 1 \\ 2 \end{bmatrix} = \begin{bmatrix} 225 \\ 225 \\ 450 \end{bmatrix} = 225 \begin{bmatrix} 1 \\ 1 \\ 2 \end{bmatrix}$$

- For $\lambda_2 = 9$, one eigenvector is $\vec v_2 = \begin{bmatrix} 1 \\ -1 \\ 0 \end{bmatrix}$, since

$$\begin{bmatrix} 42 & 33 & 75 \\ 33 & 42 & 75 \\ 75 & 75 & 150 \end{bmatrix} \begin{bmatrix} 1 \\ -1 \\ 0 \end{bmatrix} = \begin{bmatrix} 9 \\ -9 \\ 0 \end{bmatrix} = 9 \begin{bmatrix} 1 \\ -1 \\ 0 \end{bmatrix}$$

- For $\lambda_3 = 0$, one eigenvector is $\vec v_3 = \begin{bmatrix} 1 \\ 1 \\ -1 \end{bmatrix}$, since

$$\begin{bmatrix} 42 & 33 & 75 \\ 33 & 42 & 75 \\ 75 & 75 & 150 \end{bmatrix} \begin{bmatrix} 1 \\ 1 \\ -1 \end{bmatrix} = \begin{bmatrix} 0 \\ 0 \\ 0 \end{bmatrix}$$

To create $V$, all we need to do is turn $\vec v_1$, $\vec v_2$, and $\vec v_3$ into unit vectors.

$$\begin{bmatrix} 1 \\ 1 \\ 2 \end{bmatrix} \rightarrow \begin{bmatrix} \frac{1}{\sqrt{6}} \\ \frac{1}{\sqrt{6}} \\ \frac{2}{\sqrt{6}} \end{bmatrix}, \quad \begin{bmatrix} 1 \\ -1 \\ 0 \end{bmatrix} \rightarrow \begin{bmatrix} \frac{1}{\sqrt{2}} \\ -\frac{1}{\sqrt{2}} \\ 0 \end{bmatrix}, \quad \begin{bmatrix} 1 \\ 1 \\ -1 \end{bmatrix} \rightarrow \begin{bmatrix} \frac{1}{\sqrt{3}} \\ \frac{1}{\sqrt{3}} \\ -\frac{1}{\sqrt{3}} \end{bmatrix}$$

Stacking these unit vectors together gives us $V$:

$$V = \begin{bmatrix} \frac{1}{\sqrt{6}} & \frac{1}{\sqrt{2}} & \frac{1}{\sqrt{3}} \\ 
                      \frac{1}{\sqrt{6}} & -\frac{1}{\sqrt{2}} & \frac{1}{\sqrt{3}} \\ 
                      \frac{2}{\sqrt{6}} & 0 & -\frac{1}{\sqrt{3}} \end{bmatrix}$$

And indeed, since $X^TX$ is symmetric, $V$ is orthogonal: $V^TV = VV^T = I_{3 \times 3}$.

Great! We're almost done computing the SVD. So far, we have

$$\underbrace{\begin{bmatrix} 3 & 2 & 5 \\ 2 & 3 & 5 \\ 2 & -2 & 0 \\ 5 & 5 & 10 \end{bmatrix}}_X = U \underbrace{\begin{bmatrix} 15 & 0 & 0 \\ 0 & 3 & 0 \\ 0 & 0 & 0 \\ 0 & 0 & 0 \end{bmatrix}}_{\Sigma} \underbrace{\begin{bmatrix} \frac{1}{\sqrt{6}} & \frac{1}{\sqrt{6}} & \frac{2}{\sqrt{6}} \\ \frac{1}{\sqrt{2}} & -\frac{1}{\sqrt{2}} & 0 \\ \frac{1}{\sqrt{3}} & \frac{1}{\sqrt{3}} & -\frac{1}{\sqrt{3}} \end{bmatrix}}_{V^T}$$

### $XV = U \Sigma$ and $X \vec v_i = \sigma_i \vec u_i$

Ideally, we can avoid having to compute the eigenvectors of $XX^T$ to stack into $U$. And we can. If we start with

$$X = U \Sigma V^T$$

and multiply both sides on the right by $V$, we uncover a relationship between the columns of $U$ and the columns of $V$.

$$XV = U \Sigma$$

Let's unpack this. On the left, the matrix $XV$ is made up of multiplying $X$ by each column of $V$.

$$XV = X_{4 \times 3} \begin{bmatrix} | & | & | \\ \vec v_1 & \vec v_2 & \vec v_3 \\ | & | & | \end{bmatrix}_{3 \times 3} = \begin{bmatrix} | & | & | \\ X \vec v_1 & X \vec v_2 & X \vec v_3 \\ | & | & | \end{bmatrix}_{4 \times 3}$$

On the right, $U \Sigma$ is made up of stretching each column of $U$ by the corresponding singular value in the diagonal of $\mathcal{\Sigma}$.

$$U \Sigma = \begin{bmatrix} | & | & | & | \\ \vec u_1 & \vec u_2 & \vec u_3 & \vec u_4 \\ | & | & | & | \end{bmatrix}_{4 \times 4} \begin{bmatrix} \sigma_1 & 0 & 0 \\ 0 & \sigma_2 & 0 \\ 0 & 0 & \sigma_3 \\ 0 & 0 & 0 \end{bmatrix}_{4 \times 3} = \begin{bmatrix} | & | & | \\ \sigma_1 \vec u_1 & \sigma_2 \vec u_2 & \sigma_3 \vec u_3 \\ | & | & | \end{bmatrix}_{4 \times 3}$$

But, since $XV = U \Sigma$, we have

$$\begin{bmatrix} | & | & | \\ X \vec v_1 & X \vec v_2 & X \vec v_3 \\ | & | & | \end{bmatrix} = \begin{bmatrix} | & | & | \\ \sigma_1 \vec u_1 & \sigma_2 \vec u_2 & \sigma_3 \vec u_3 \\ | & | & | \end{bmatrix}$$

:::{attention} Singular values represent stretches, too!
A consequence of the above relationship is that if $X$ is $n \times d$, then for $i = 1, 2, ..., r$, where $r = \text{rank}(X)$,

$$X \vec v_i = \sigma_i \vec u_i$$

**The above is saying that when $X$ is multiplied by $\vec v_i$, the result is a scaled version of $\vec u_i$ (not $\vec v_i$)!** 

Remember, if $X$ is $n \times d$, then $\vec v_i \in \mathbb{R}^d$ and $X \vec v_i \in \mathbb{R}^n$. All that is to say, $X \vec v_i$ is a vector in a different dimension than $\vec v_i$, so it can't be the case that $X \vec v_i$ is in the same direction as $\vec v_i$. Instead, $X \vec v_i$ is a vector in the same direction as $\vec u_i$, which lives in $\mathbb{R}^n$ like $X \vec v_i$ does.

$X \vec v_i = \sigma_i \vec u_i$ is the singular value/vector analog of $A \vec v = \lambda \vec v$ for eigenvalues/eigenvectors.
:::

**Crucially though**, $X \vec v_i = \sigma_i \vec u_i$ only holds when we've arranged the singular values and vectors in $U$, $\mathcal{\Sigma}$, and $V^T$ consistently. This is one reason why we always arrange the singular values in $\mathcal{\Sigma}$ in decreasing order.

Back to our example. Again, we currently have

$$\underbrace{\begin{bmatrix} 3 & 2 & 5 \\ 2 & 3 & 5 \\ 2 & -2 & 0 \\ 5 & 5 & 10 \end{bmatrix}}_X = U \underbrace{\begin{bmatrix} 15 & 0 & 0 \\ 0 & 3 & 0 \\ 0 & 0 & 0 \\ 0 & 0 & 0 \end{bmatrix}}_{\Sigma} \underbrace{\begin{bmatrix} \frac{1}{\sqrt{6}} & \frac{1}{\sqrt{6}} & \frac{2}{\sqrt{6}} \\ \frac{1}{\sqrt{2}} & -\frac{1}{\sqrt{2}} & 0 \\ \frac{1}{\sqrt{3}} & \frac{1}{\sqrt{3}} & -\frac{1}{\sqrt{3}} \end{bmatrix}}_{V^T}$$

We know $X$, and we know each $\vec v_i$ and $\sigma_i$. Rearranging $X \vec v_i = \sigma_i \vec u_i$ gives us

$$\vec u_i = \frac{1}{\sigma_i} X \vec v_i$$

which is a recipe for computing $\vec u_1$, $\vec u_2$, and $\vec u_3$.

1. $\vec u_1 = \frac{1}{15} X \begin{bmatrix} \frac{1}{\sqrt{6}} \\ \frac{1}{\sqrt{6}} \\ \frac{2}{\sqrt{6}} \end{bmatrix} = \begin{bmatrix} \frac{1}{\sqrt{6}} \\ \frac{1}{\sqrt{6}} \\ 0 \\ \frac{2}{\sqrt{6}} \end{bmatrix}$.
1. $\vec u_2 = \frac{1}{3} X \begin{bmatrix} \frac{1}{\sqrt{2}} \\ -\frac{1}{\sqrt{2}} \\ 0 \end{bmatrix} = \begin{bmatrix} \frac{1}{3\sqrt{2}} \\ -\frac{1}{3\sqrt{2}} \\ \frac{2\sqrt{2}}{3} \\ 0 \end{bmatrix}$
1. $\vec u_3 = \frac{1}{0} X \begin{bmatrix} \frac{1}{\sqrt{3}} \\ \frac{1}{\sqrt{3}} \\ -\frac{1}{\sqrt{3}} \end{bmatrix} = ... \text{wait, what?}$

Something is not quite right: $\sigma_3 = 0$, which means we can't use $\vec u_i = \frac{1}{\sigma_i} X \vec v_i$ to compute $\vec u_3$. And, even if we could, this recipe tells us nothing about $\vec u_4$, which we also need to find.

### Null Spaces Return

The issue is that we don't have a recipe for what $\vec u_3$ and $\vec u_4$ should be. This problem stems from the original matrix $X$ only having a rank of 2:

$$X = \begin{bmatrix} 3 & 2 & 5 \\ 2 & 3 & 5 \\ 2 & -2 & 0 \\ 5 & 5 & 10 \end{bmatrix}$$

TODO discuss how $\vec u_3$ and $\vec u_4$ are in the null space of $X$, but for $U$ to be orthogonal, they should be orthogonal to each other, and thus be a basis for the null space of $X$. Similarly, $\vec v_3$ is a basis for the null space of $X^TX$.

### Conclusion

Put a definitive picture about what the vectors in each $U$ and $V$ are bases for.

Something about the minus signs somewhere

### The roles of the zeros

---

## The Rank One Decomposition

$X = U \Sigma V^T$

### Compact SVD

The version of the SVD we've worked with so far, where $U$ is $n \times n$, $\mathcal{\sigma}$ is $n \times d$, and $V^T$ is $d \times d$, is called the **full SVD**. It contains all of the information one needs to recreate $X$, but it contains a lot of extra 0's that don't aid in computation.

Instead, the **compact SVD** only contains the non-zero singular values and the corresponding singular vectors.

### A Sum of Rank One Matrices

Let $r = \text{rank}(X)$. Then, the SVD of $X$ can be written as

$$X = U \Sigma V^T = \sigma_1 \vec u_1 \vec v_1^T + \sigma_2 \vec u_2 \vec v_2^T + \cdots + \sigma_r \vec u_r \vec v_r^T$$

Each term $\sigma_i \vec u_i \vec v_i^T$ is a rank one matrix. Each column of $\sigma_i \vec u_i \vec v_i^T$ is a multiple of $\sigma_i \vec u_i$.

### The Best Low Rank Approximation



---

## Rank One Pieces

Put the flag animation

---

## Computing the SVD

The invariance of the negative signs. Not actually computing $X^TX$ or $XX^T$.

In [150]:
X = np.array([[3, 2, 5],
              [2, 3, 5],
              [2, -2, 0],
              [5, 5, 10]])
u, s, vt = np.linalg.svd(X)
u.shape, s.shape, vt.shape

((4, 4), (3,), (3, 3))

In [152]:
u

array([[-4.08248290e-01,  2.35702260e-01, -4.57353982e-01,
        -7.54059091e-01],
       [-4.08248290e-01, -2.35702260e-01, -6.80988664e-01,
         5.60385775e-01],
       [-9.34078423e-17,  9.42809042e-01, -5.59086707e-02,
         3.28611217e-01],
       [-8.16496581e-01, -1.15075004e-16,  5.69171323e-01,
         9.68366582e-02]])

In [155]:
s[1]

2.9999999999999982

Notice that `u` is a $4 \times 4$ matrix and `vt` is a $3 \times 3$ matrix, as promised. It returns back `s` as a 1-dimensional array rather than a diagonal matrix, which we'll need to reshape to work with.

In [151]:
u

array([[-4.08248290e-01,  2.35702260e-01, -4.57353982e-01,
        -7.54059091e-01],
       [-4.08248290e-01, -2.35702260e-01, -6.80988664e-01,
         5.60385775e-01],
       [-9.34078423e-17,  9.42809042e-01, -5.59086707e-02,
         3.28611217e-01],
       [-8.16496581e-01, -1.15075004e-16,  5.69171323e-01,
         9.68366582e-02]])

In [148]:
vt

array([[-4.08248290e-01, -4.08248290e-01, -8.16496581e-01],
       [ 7.07106781e-01, -7.07106781e-01,  1.23697461e-16],
       [-5.77350269e-01, -5.77350269e-01,  5.77350269e-01]])

In [114]:
u

array([[-7.07106781e-01,  2.35702260e-01, -6.66666667e-01],
       [-7.07106781e-01, -2.35702260e-01,  6.66666667e-01],
       [-2.99816808e-16,  9.42809042e-01,  3.33333333e-01]])

In [88]:
print(u, '\n')
print(s, '\n')
print(vt, '\n')

[[ 0.83628634 -0.54829295  0.          0.        ]
 [-0.52015636 -0.79337088  0.31622777  0.        ]
 [ 0.          0.          0.          1.        ]
 [ 0.17338545  0.26445696  0.9486833   0.        ]] 

[1.21022961e+01 2.92137448e+00 2.51214793e-16] 

[[ 0.3455073   0.41967242  0.83934484]
 [-0.93841606  0.15451556  0.30903113]
 [-0.          0.89442719 -0.4472136 ]] 



### Sum of Rank One Matrices

An observation you may have seen above is that only the first $r$ columns of $U$ and the first $r$ columns of $V$ are directly useful.

---

## Key Takeaways

- something about $A \vec v_i = \sigma_i \vec u_i$